# 1. 단일 기업 재무제표 받기

In [2]:
import sys
import pandas as pd
import time

# 1. 모듈 임포트 및 경로 설정
sys.path.append(r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\collect")

from sec_data_pipeline.collectors.get_us_ticker import get_filtered_us_tickers
from sec_data_pipeline.collectors.rate_limiter import AdaptiveRateLimiter # 반드시 확인
from sec_data_pipeline.valuation.integrated_financial_analyzer_mysql_fixed import IntegratedFinancialAnalyzer
from sec_data_pipeline.storage.db_manager import DBManager
from DATA.stock_invest_function import get_db_host

# ---------------------------------------------------------
# [설정 항목] 여기서 날짜와 개수를 조절하세요
# ---------------------------------------------------------
START_DATE = "2025-09-01"  # ✅ 이 날짜 이후의 데이터만 수집/저장
MAX_TICKERS = 5000         # ✅ 테스트로 몇 개만 할지 결정 (전체는 None 또는 큰 숫자)
OFFSET = 0              # ✅ 시작 위치 (4000번 인덱스부터)
# ---------------------------------------------------------

# 2. DB 및 객체 초기화
db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}
db_manager = DBManager(db_info)
analyzer = IntegratedFinancialAnalyzer(db_info=db_info, wrds_conn_str="여기에_WRDS_주소")

# ✅ [중요] NameError 방지를 위한 rate_limiter 정의
rate_limiter = AdaptiveRateLimiter(
    max_calls=8,
    time_window=1.0,
    min_calls=3,
    backoff_factor=0.8,
)

headers = {"User-Agent": "Hoyoung Research <stox1224@gmail.com>"}

# 3. 티커 리스트 준비 (슬라이싱 적용)
ALL_TICKERS = get_filtered_us_tickers()
TICKER_LIST = ALL_TICKERS[OFFSET : OFFSET + MAX_TICKERS]

print(f"총 {len(TICKER_LIST)}개의 티커를 수집합니다. (기준일: {START_DATE})")

# 4. 메인 루프
for i, ticker in enumerate(TICKER_LIST, start=1):
    print(f"\n=== [{i}/{len(TICKER_LIST)}] {ticker} 처리 시작 ===")

    # 이제 정의된 rate_limiter를 사용하므로 에러가 나지 않습니다.
    rate_limiter.wait_if_needed()

    try:
        result = analyzer.analyze(ticker, headers=headers, table_name="us_fundq")
        if result is None:
            continue

        final_df, cik, entity_name = result

        # ✅ [추가] 날짜 필터링 로직 적용
        if not final_df.empty:
            final_df.index = pd.to_datetime(final_df.index)
            # 설정한 START_DATE 이후 데이터만 필터링
            final_df = final_df[final_df.index >= pd.to_datetime(START_DATE)]

        if final_df.empty:
            print(f"⚠ {ticker}: {START_DATE} 이후의 새로운 데이터가 없어 저장을 건너뜁니다.")
            continue

        # DB 저장용 인덱스 정리
        if final_df.index.name != "date":
            final_df.index.name = "date"

        db_manager.save_normalized_data(
            ticker=ticker,
            cik=cik,
            df=final_df,
            item_mapping=None
        )
        print(f"✓ {ticker} ({entity_name}) {len(final_df)}건 저장 완료")

    except Exception as e:
        msg = str(e)
        if "429" in msg:
            print(f"⚠ {ticker}: SEC 429 감지 → 60초 대기")
            rate_limiter.on_rate_limit_error()
            time.sleep(60)
            continue
        print(f"✗ {ticker} 처리 중 오류: {e}")
        continue

print("\n=== 작업 완료 ===")


✓ DB connected: 192.168.0.230:3307/investar


100%|██████████| 298/298 [00:00<00:00, 919.76it/s] 


제외된 기업 수: 1808
남은 기업 수: 4987
티커 수: 4987
총 4987개의 티커를 수집합니다. (기준일: 2025-09-01)

=== [1/4987] NVDA 처리 시작 ===
[NVDA] 통합 재무분석 시작
✓ Entity: NVIDIA CORP (CIK: 1045810)
✓ EDGAR 정규화 DF: 74 rows × 32 cols
✓ MySQL 연결 성공: 192.168.0.230:3307/investar
⚠ WRDS 쿼리 실패: Execution failed on sql 'SELECT * FROM us_fundq WHERE tic=%s': (1146, "Table 'investar.us_fundq' doesn't exist")
WRDS 데이터 없음 → EDGAR 그대로 반환
✓ EDGAR+WRDS 병합 DF: 74 rows × 32 cols
재무비율 계산 중...
  - 수익성 비율 계산...
  - 레버리지 비율 계산...
  - 유동성 비율 계산...
  - 효율성 비율 계산...
재무비율 계산 완료!
✓ 최종 DF (비율 포함): 74 rows × 49 cols
[NVDA] 통합 재무분석 종료
✓ Saved NVDA data: 1 dates x 49 items
✓ NVDA (NVIDIA CORP) 1건 저장 완료

=== [2/4987] AAPL 처리 시작 ===
[AAPL] 통합 재무분석 시작
✓ Entity: Apple Inc. (CIK: 320193)
✓ EDGAR 정규화 DF: 72 rows × 32 cols
⚠ WRDS 쿼리 실패: Execution failed on sql 'SELECT * FROM us_fundq WHERE tic=%s': (1146, "Table 'investar.us_fundq' doesn't exist")
WRDS 데이터 없음 → EDGAR 그대로 반환
✓ EDGAR+WRDS 병합 DF: 72 rows × 32 cols
재무비율 계산 중...
  - 수익성 비율 계산...
  - 레버리지 비율 계산..

In [4]:
import pandas as pd
import pymysql
from typing import Dict, Optional, List


def quarter_end_from_report_date(report_date: pd.Timestamp) -> pd.Timestamp:
    y = report_date.year
    m = report_date.month

    if m in [1, 2, 3]:
        return pd.Timestamp(f"{y}-03-31")
    elif m in [4, 5, 6]:
        return pd.Timestamp(f"{y}-06-30")
    elif m in [7, 8, 9]:
        return pd.Timestamp(f"{y}-09-30")
    else:
        return pd.Timestamp(f"{y}-12-31")


def fetch_sec_financial_pivot_fixed(
    db_info: Dict[str, any],
    ticker: str,
    table_name: str = "sec_financial_data",
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    item_list: Optional[List[str]] = None,
) -> pd.DataFrame:

    conn = pymysql.connect(
        host=db_info["host"], port=db_info.get("port", 3306),
        user=db_info["user"], password=db_info["password"],
        database=db_info["database"], charset="utf8mb4"
    )

    # -----------------------------
    # 1) 데이터 로드
    # -----------------------------
    try:
        where_clause = ["ticker=%s"]
        params = [ticker]

        if start_date:
            where_clause.append("date >= %s")
            params.append(start_date)
        if end_date:
            where_clause.append("date <= %s")
            params.append(end_date)
        if item_list:
            placeholders = ",".join(["%s"] * len(item_list))
            where_clause.append(f"item_name IN ({placeholders})")
            params.extend(item_list)

        where_sql = " AND ".join(where_clause)

        sql = f"""
            SELECT date, ticker, item_name, value
            FROM {table_name}
            WHERE {where_sql}
            ORDER BY date, item_name
        """

        df = pd.read_sql(sql, conn, params=params)
    finally:
        conn.close()

    if df.empty:
        print(f"[INFO] {ticker} 데이터 없음")
        return df

    # -----------------------------
    # 2) 기존 date → report_date로 이름 변경
    # -----------------------------
    df = df.rename(columns={"date": "report_date"})
    df["report_date"] = pd.to_datetime(df["report_date"])

    # -----------------------------
    # 3) report_date → 분기말 날짜(date)
    # -----------------------------
    df["date"] = df["report_date"].apply(quarter_end_from_report_date)

    # (중복 가능성 대비) 각 분기(date, ticker)별 대표 report_date 하나 선택
    # 여기서는 가장 늦은 날짜(max) 사용
    rep_dates = (
        df[["date", "ticker", "report_date"]]
        .drop_duplicates()
        .groupby(["date", "ticker"], as_index=False)["report_date"]
        .max()
    )

    # -----------------------------
    # 4) pivot 변환 (분기말 기준 wide)
    # -----------------------------
    df_pivot = df.pivot_table(
        index=["date", "ticker"],
        columns="item_name",
        values="value",
        aggfunc="last"
    ).sort_index()

    df_pivot = df_pivot.reset_index()

    # -----------------------------
    # 5) 대표 report_date 다시 붙이기
    # -----------------------------
    df_pivot = df_pivot.merge(rep_dates, how="left", on=["date", "ticker"])

    # 보기 편하게 컬럼 순서 조정: date, report_date, ticker, 나머지 item 들
    cols = ["period_end", "ticker"] + [c for c in df_pivot.columns if c not in ("period_end","ticker")]
    df_pivot = df_pivot[cols]
    df_pivot = df_pivot[cols]

    return df_pivot


In [10]:
# 1) AAPL 전체 기간, 모든 item_name
fs_df = fetch_sec_financial_pivot_fixed(db_info, "A")
fs_df

# 2) 기간 제한 + 특정 항목만
# item_list = ["revenue", "net_income", "total_assets", "ROE", "OPM"]
# aapl_subset = fetch_sec_financial_pivot(
#     db_info,
#     "AAPL",
#     start_date="2015-01-01",
#     end_date="2024-12-31",
#     item_list=item_list
# )
# print(aapl_subset.tail())

,date,report_date,ticker,accounts_payable,accounts_receivable,accrued_liabilities,accumulated_depreciation,asset_turnover,capital_expenditures,cash,...,receivables_turnover,research_development,revenue,roa,roe,roic,short_term_debt,stockholders_equity,total_assets,total_liabilities
0,2006-12-31,2006-10-31,A,NaN,NaN,NaN,NaN,NaN,NaN,2.262000e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.648000e+09,NaN,NaN
1,2007-12-31,2007-10-31,A,NaN,NaN,NaN,NaN,NaN,NaN,1.826000e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.234000e+09,NaN,NaN
2,2008-09-30,2008-07-31,A,NaN,NaN,NaN,NaN,NaN,NaN,1.640000e+09,...,NaN,170000000.0,NaN,NaN,NaN,NaN,NaN,3.234000e+09,NaN,NaN
3,2008-12-31,2008-10-31,A,308000000.0,7.700000e+08,409000000.0,NaN,NaN,NaN,1.405000e+09,...,NaN,170000000.0,NaN,NaN,NaN,4.851270,NaN,2.559000e+09,7.007000e+09,4.448000e+09
4,2009-03-31,2009-01-31,A,308000000.0,7.700000e+08,409000000.0,NaN,NaN,34000000.0,1.362000e+09,...,NaN,169000000.0,NaN,NaN,NaN,0.534084,NaN,2.559000e+09,7.007000e+09,4.448000e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,2024-09-30,2024-07-31,A,497000000.0,1.227000e+09,309000000.0,1.223000e+09,0.145632,90000000.0,1.779000e+09,...,1.22993,127000000.0,1.578000e+09,9.68114,18.3056,3.321140,795000000.0,5.903000e+09,1.099600e+10,5.093000e+09
67,2024-12-31,2024-10-31,A,540000000.0,1.324000e+09,368000000.0,1.304000e+09,0.139590,90000000.0,1.329000e+09,...,1.20688,127000000.0,1.578000e+09,10.79220,20.7783,3.183660,45000000.0,5.898000e+09,1.184600e+10,5.948000e+09
68,2025-03-31,2025-01-31,A,547000000.0,1.328000e+09,258000000.0,1.304000e+09,0.147056,97000000.0,1.467000e+09,...,1.28174,113000000.0,1.681000e+09,10.41030,19.4842,3.732370,16000000.0,6.027000e+09,1.191400e+10,5.887000e+09
69,2025-06-30,2025-04-30,A,517000000.0,1.366000e+09,347000000.0,1.304000e+09,0.144955,97000000.0,1.486000e+09,...,1.27572,112000000.0,1.668000e+09,9.53333,17.7652,2.812820,146000000.0,6.136000e+09,1.215800e+10,6.022000e+09


In [4]:
def count_unique_tickers_sec_financial(db_info: dict,
                                       table_name: str = "sec_financial_data") -> int:
    """
    sec_financial_data 테이블에서 unique ticker 개수를 조회하여 반환하는 함수.
    """
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        database=db_info["database"],
        charset="utf8mb4"
    )

    try:
        query = f"SELECT DISTINCT ticker FROM {table_name};"
        df = pd.read_sql(query, conn)

        unique_ticker_count = df["ticker"].nunique()

        print(f"[INFO] Unique tickers in {table_name}: {unique_ticker_count}")
        return unique_ticker_count

    finally:
        conn.close()

In [6]:
import pymysql

count_unique_tickers_sec_financial(db_info)

[INFO] Unique tickers in sec_financial_data: 1571


1571

In [25]:
aapl_df.columns.tolist()

['date',
 'ticker',
 'accounts_payable',
 'accounts_receivable',
 'accrued_liabilities',
 'accumulated_depreciation',
 'asset_turnover',
 'capital_expenditures',
 'cash',
 'cost_of_revenue',
 'current_assets',
 'current_liabilities',
 'current_ratio',
 'days_inventory',
 'days_receivables',
 'debt_to_assets',
 'debt_to_equity',
 'deferred_revenue',
 'depreciation_amortization',
 'equity_multiplier',
 'goodwill',
 'gross_margin',
 'gross_profit',
 'income_tax_expense',
 'intangible_assets',
 'interest_expense',
 'interest_income',
 'inventory',
 'inventory_turnover',
 'long_term_debt',
 'long_term_debt_current',
 'net_income',
 'net_margin',
 'net_ppe',
 'operating_expenses',
 'operating_income',
 'operating_margin',
 'other_noncurrent_assets',
 'pretax_income',
 'quick_ratio',
 'receivables_turnover',
 'research_development',
 'revenue',
 'roa',
 'roe',
 'roic',
 'short_term_debt',
 'stockholders_equity',
 'total_assets',
 'total_liabilities']

In [ ]:
# # 1) EDGAR 수집/정규화
# headers = {"User-Agent": "HoyoungPark Research <stox1224@email.com>"}
# facts   = fetch_company_facts( TICKER, headers=headers)
# parser  = CompanyFactsParser(facts)
# normal  = FinancialNormalizer(parser)
# edgar_df = normal.create_normalized_dataframe(period_type="quarterly")
#
# # 2) WRDS Validator
# db_info = {
#     "host": get_db_host(),
#     "port": 3307,
#     "user": "stox7412",
#     "password": "Apt106503!~",
#     "database": "investar",
# }
# validator = WRDSDataValidator(db_info)
#
# import sqlalchemy as sa
#
# # ① EDGAR 분기 DF 준비 (이미 갖고 계신 df: edgar_df)
# #    edgar_df.index = 분기 날짜, 컬럼에 'revenue' 포함
#
# # ② WRDS(us_fundq)에서 AAPL의 edate, ticker, saleq 로드
# db_info = {
#     "host": "192.168.0.230",
#     "port": 3307,
#     "user": "stox7412",
#     "password": "Apt106503!~",
#     "database": "investar",
# }


In [3]:
tickers = get_filtered_us_tickers()

100%|██████████| 294/294 [00:00<00:00, 1034.88it/s]


제외된 기업 수: 1812
남은 기업 수: 5001
티커 수: 5001


In [5]:
ticker_list = tickers[:10]

['NVDA',
 'AAPL',
 'MSFT',
 'AMZN',
 'GOOGL',
 'AVGO',
 'GOOG',
 'META',
 'TSLA',
 'NFLX']

In [6]:
# import pandas as pd
#
# # 1. 클라이언트 및 다운로더 설정
# user_agent = "PersonalResearch stox1224@email.com"
# client = SECAPIClient(user_agent, RateLimiter(10, 1.0))
# downloader = BulkDownloader(client, output_dir="./sec_data", max_workers=3)
#
# # 2. 다운로드할 기업 리스트
# # tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'META']
#
# # 3. 배치 다운로드 (자동으로 JSON 파일 저장)
# results = downloader.download_company_facts_batch(ticker_list)
#
# # 4. 각 기업별 데이터 처리
# all_financials = {}
#
# for ticker, company_facts in results.items():
#     headers = {"User-Agent": "HoyoungPark Research <stox1224@email.com>"}
#     facts   = fetch_company_facts(ticker, headers=headers)
#     parser  = CompanyFactsParser(facts)
#     normal  = FinancialNormalizer(parser)
#     edgar_df = normal.create_normalized_dataframe(period_type="quarterly")
#
#
#     parser = CompanyFactsParser(company_facts)
#     normalizer = FinancialNormalizer(parser)
#
#     # 재무데이터 정규화
#     df = normalizer.create_normalized_dataframe(period_type='quarterly')
#     df_billions = normalizer.convert_to_billions(df)
#
#     all_financials[ticker] = df_billions
#
#     # 개별 CSV 저장
#     # normalizer.export_to_csv(df_billions, f'{ticker}_financials.csv')
#
# print(f"총 {len(all_financials)}개 기업 데이터 수집 완료")

NameError: name 'SECAPIClient' is not defined

In [21]:
TICKER = "GLW"   # ← 여기만 바꾸면 전체가 따라옵니

# 1) EDGAR 수집/정규화
headers = {"User-Agent": "HoyoungPark Research <stox1224@email.com>"}
facts   = fetch_company_facts( TICKER, headers=headers)
parser  = CompanyFactsParser(facts)
normal  = FinancialNormalizer(parser)
edgar_df = normal.create_normalized_dataframe(period_type="quarterly")

# 2) WRDS Validator
db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}
validator = WRDSDataValidator(db_info)

# 3) 결합/보완 실행 (핵심: table_name='us_fundq', date_col='edate')
result_df = validator.validate_and_fill_improved(
    edgar_df=edgar_df,
    ticker= TICKER,
    table_name="us_fundq",   # 당신 DB 테이블
    days_tolerance=15,
    verbose=True,
    ticker_col="ticker",     # 당신 DB 컬럼
    date_col="edate"         # 당신 DB 컬럼
)

import sqlalchemy as sa

# ① EDGAR 분기 DF 준비 (이미 갖고 계신 df: edgar_df)
#    edgar_df.index = 분기 날짜, 컬럼에 'revenue' 포함

# ② WRDS(us_fundq)에서 AAPL의 edate, ticker, saleq 로드
db_info = {
    "host": "192.168.0.230",
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}
df_wrds = fetch_wrds_fundq_sample(
    db_info, ticker=TICKER , columns=["edate", "ticker", "saleq"], table_name="US_fundq"
)

# ③ 결측 보완 + 단위 정합
filled_df = fill_revenue_with_wrds(
    edgar_df=edgar_df,
    wrds_df=df_wrds,
    wrds_value_col="saleq",     # WRDS의 분기 매출 컬럼명
    days_tolerance=20,          # 분기말 근접 허용일
    verbose=True
)

print("\n[CHECK] revenue NaN:")
print("  before:", edgar_df["revenue"].isna().sum())
print("  after :", filled_df["revenue"].isna().sum())

if filled_df.empty:
    print("⚠ filled_df가 비어 있습니다 → result_df를 target_df로 대체합니다.")
    target_df = result_df.copy()
else:
    print("✓ filled_df가 유효합니다 → target_df에 filled_df를 사용합니다.")
    target_df = filled_df.copy()

from sec_data_pipeline.parsers.financial_normalizer import FinancialNormalizer


# parser는 EDGAR 데이터를 읽어온 CompanyFactsParser 등의 인스턴스
normalizer = FinancialNormalizer(parser=None)  # parser가 필요 없으면 None으로 두세요


# 1) 기본 추정: income_tax_expense / pretax_income
est = (filled_df.get('income_tax_expense') / filled_df.get('pretax_income'))

# 2) 이상치/음수/무한대 정리
est = est.replace([np.inf, -np.inf], np.nan)
# 적자 구간(pretax_income <= 0)은 추정치 제거
est = est.mask((filled_df.get('pretax_income') <= 0), np.nan)
# 0~0.5로 클리핑(50% 이상은 보통 이상치)
est = est.clip(lower=0.0, upper=0.3)

# 3) 결측 보강: 최근 값으로 보간 + 기본값 대체
# est = est.ffill().fillna(0.21)

# filled_df['tax_rate'] = 0.21
ratio = normal.calculate_financial_ratios(target_df)


['NVDA',
 'AAPL',
 'MSFT',
 'AMZN',
 'GOOGL',
 'AVGO',
 'GOOG',
 'META',
 'TSLA',
 'NFLX',
 'COST',
 'PLTR',
 'ASML',
 'AMD',
 'CSCO',
 'AZN',
 'MU',
 'TMUS',
 'ISRG',
 'SHOP',
 'PEP',
 'AMAT',
 'LRCX',
 'LIN',
 'APP',
 'AMGN',
 'QCOM',
 'INTC',
 'PDD',
 'BKNG',
 'GILD',
 'KLAC',
 'TXN',
 'ARM',
 'ADBE',
 'PANW',
 'CRWD',
 'ADI',
 'SNY',
 'HON',
 'VRTX',
 'MELI',
 'ADP',
 'SBUX',
 'CMCSA',
 'NTES',
 'ORLY',
 'DASH',
 'REGN',
 'CDNS',
 'MAR',
 'SNPS',
 'CTAS',
 'MNST',
 'MDLZ',
 'MRVL',
 'ABNB',
 'CSX',
 'ADSK',
 'WDAY',
 'IDXX',
 'FTNT',
 'TRI',
 'ROST',
 'PYPL',
 'STX',
 'WBD',
 'ALNY',
 'ARGX',
 'DDOG',
 'PCAR',
 'WDC',
 'EA',
 'MSTR',
 'BKR',
 'NXPI',
 'ROP',
 'FER',
 'JD',
 'ZS',
 'FAST',
 'TCOM',
 'SYM',
 'TTWO',
 'INSM',
 'MPWR',
 'FANG',
 'AXON',
 'CCEP',
 'BIDU',
 'PAYX',
 'TEAM',
 'CPRT',
 'EBAY',
 'CTSH',
 'ONC',
 'KDP',
 'GEHC',
 'CRWV',
 'RYAAY',
 'KMB',
 'FISV',
 'NTRA',
 'SNDK',
 'UAL',
 'EXPE',
 'VRSK',
 'KHC',
 'CSGP',
 'ERIC',
 'VOD',
 'TSCO',
 'ODFL',
 'MCHP',
 'FSLR'

### Billions 단위로 변환

In [ ]:
df_billions = normalizer.convert_to_billions(df)
print(df_billions[['revenue', 'net_income']].tail(8))

### TTM (Trailing Twelve Months) 계산

In [ ]:
df_ttm = normalizer.calculate_ttm(df_billions, columns=['revenue', 'net_income'])
print(df_ttm[['revenue', 'revenue_ttm']].tail(8))

# 2. 다수 기업 재무제표 받기

In [ ]:
from collectors import SECAPIClient, RateLimiter, BulkDownloader
from parsers import CompanyFactsParser, FinancialNormalizer
import pandas as pd

# 1. 클라이언트 및 다운로더 설정
user_agent = "PersonalResearch stox1224@email.com"
client = SECAPIClient(user_agent, RateLimiter(10, 1.0))
downloader = BulkDownloader(client, output_dir="./sec_data", max_workers=3)

# 2. 다운로드할 기업 리스트
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'META']

# 3. 배치 다운로드 (자동으로 JSON 파일 저장)
results = downloader.download_company_facts_batch(tickers)

# 4. 각 기업별 데이터 처리
all_financials = {}

for ticker, company_facts in results.items():
    parser = CompanyFactsParser(company_facts)
    normalizer = FinancialNormalizer(parser)

    # 재무데이터 정규화
    df = normalizer.create_normalized_dataframe(period_type='quarterly')
    df_billions = normalizer.convert_to_billions(df)

    all_financials[ticker] = df_billions

    # 개별 CSV 저장
    normalizer.export_to_csv(df_billions, f'{ticker}_financials.csv')

print(f"총 {len(all_financials)}개 기업 데이터 수집 완료")

# 3. 특정 재무항목만 추출

In [ ]:
revenue_comparison = {}

for ticker, company_facts in results.items():
    parser = CompanyFactsParser(company_facts)
    normalizer = FinancialNormalizer(parser)

    # Revenue만 추출
    revenue_series = normalizer.normalize_single_item('revenue', period_type='quarterly')

    if revenue_series is not None:
        revenue_comparison[ticker] = revenue_series / 1_000_000_000  # Billions

# DataFrame으로 변환
revenue_df = pd.DataFrame(revenue_comparison)
print("\n최근 8분기 매출 비교 (Billions):")
print(revenue_df.tail(8))

# YoY 성장률
growth_df = revenue_df.pct_change(periods=4) * 100
print("\nYoY 매출 성장률 (%):")
print(growth_df.tail(4))

# 4. 사용 가능한 재무 항목


In [ ]:
# 추출 가능한 표준 재무 항목들
available_items = [
    'revenue',              # 매출
    'net_income',           # 순이익
    'operating_income',     # 영업이익
    'gross_profit',         # 매출총이익
    'total_assets',         # 총자산
    'current_assets',       # 유동자산
    'total_liabilities',    # 총부채
    'current_liabilities',  # 유동부채
    'stockholders_equity',  # 자본
    'cash',                 # 현금
    'long_term_debt',       # 장기부채
    'cost_of_revenue',      # 매출원가
    'operating_expenses',   # 영업비용
    'research_development', # 연구개발비
    'earnings_per_share',   # 주당순이익
]

# 특정 항목만 추출
for item in ['revenue', 'net_income', 'total_assets']:
    series = normalizer.normalize_single_item(item, period_type='quarterly')
    print(f"\n{item}:")
    print(series.tail(4))

# 5. 재무비율 계산

In [ ]:
# 데이터 준비
df = normalizer.create_normalized_dataframe(period_type='quarterly')

# 재무비율 자동 계산
df_ratios = normalizer.calculate_financial_ratios(df)

# 계산되는 비율들:
# - profit_margin: 순이익률
# - operating_margin: 영업이익률
# - current_ratio: 유동비율
# - debt_to_equity: 부채비율
# - roe: 자기자본이익률
# - roa: 총자산이익률

print(df_ratios[['revenue', 'net_income', 'profit_margin']].tail(8))

# 6. 성장률 분석

In [ ]:
df_billions = normalizer.convert_to_billions(df)

# YoY 성장률 계산
df_growth = normalizer.calculate_growth_rates(
    df_billions,
    columns=['revenue', 'net_income'],
    periods=4  # 4분기 = YoY
)

print(df_growth[['revenue', 'revenue_yoy_growth']].tail(8))

# 7. 진행상황 모니터링

In [ ]:
# 콜백 함수 정의
def on_download_complete(ticker, data, index, total):
    if data:
        entity_name = data.get('entityName', 'Unknown')
        print(f"[{index}/{total}] {ticker}: {entity_name} - 완료")
    else:
        print(f"[{index}/{total}] {ticker}: 실패")

# 콜백과 함께 다운로드
results = downloader.download_company_facts_batch(
    tickers=['AAPL', 'MSFT', 'GOOGL'],
    callback=on_download_complete
)

# 8. 중단된 다운로드 재개


In [ ]:
# 이미 다운로드된 파일은 건너뛰기
remaining_results = downloader.resume_download(tickers)

# 9. 다운로드 리포트 생성

In [ ]:
# 다운로드 후 리포트 생성
downloader.create_download_report(results, "download_report.json")

# 10. 실전 예제: S&P 500 상위 기업 분석

In [ ]:
# Tech Giants
tech_giants = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA']

# 다운로드
downloader = BulkDownloader(client, output_dir="./tech_giants", max_workers=5)
results = downloader.download_company_facts_batch(tech_giants)

# Revenue 비교 분석
revenue_data = {}
for ticker, facts in results.items():
    parser = CompanyFactsParser(facts)
    normalizer = FinancialNormalizer(parser)
    revenue = normalizer.normalize_single_item('revenue', period_type='quarterly')
    if revenue is not None:
        revenue_data[ticker] = revenue / 1e9  # Billions

# 비교 DataFrame
comparison = pd.DataFrame(revenue_data)
print("\nTech Giants 최근 매출 비교 (Billions):")
print(comparison.tail(8))

# 시각화 (선택사항)
import matplotlib.pyplot as plt
comparison.tail(12).plot(figsize=(12, 6))
plt.title('Tech Giants Revenue Comparison (Last 12 Quarters)')
plt.ylabel('Revenue (Billions USD)')
plt.legend(loc='best')
plt.grid(True)
plt.savefig('tech_giants_revenue.png')